<a href="https://colab.research.google.com/github/raksaadhinata843/text-to-sql_LLM_repo_low_cost/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boto3

In [ ]:
import json
import os
import time
import boto3
import requests

def get_all_nasdaq_ciks(user_email):
    """Mengambil semua daftar CIK yang khusus terdaftar di bursa NASDAQ"""
    # CORRECT ENDPOINT: SEC mapping for tickers and exchanges
    ticker_url = "https://www.sec.gov/files/company_tickers_exchange.json"
    headers = {
        "User-Agent": f"RaksaProject ({user_email})",
        "Accept-Encoding": "gzip, deflate",
    }

    try:
        response = requests.get(ticker_url, headers=headers)
        response.raise_for_status()
        raw_data = response.json()

        fields = raw_data["fields"]
        cik_idx = fields.index("cik")
        exchange_idx = fields.index("exchange")

        nasdaq_ciks = []
        for row in raw_data["data"]:
            if str(row[exchange_idx]).lower() == "nasdaq":
                cik_padded = str(row[cik_idx]).zfill(10)
                nasdaq_ciks.append(cik_padded)

        return nasdaq_ciks

    except Exception as e:
        print(f"Gagal mengambil daftar CIK Nasdaq: {str(e)}")
        return []

def fetch_and_save_to_s3(cik, user_email, bucket_name):
    company_data_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    headers = {
        "User-Agent": f"RaksaProject ({user_email})",
        "Accept-Encoding": "gzip, deflate",
    }

    response = requests.get(company_data_url, headers=headers)
    response.raise_for_status()
    data = response.json()

    tickers = data.get("tickers", [])
    ticker = tickers[0] if tickers else "UNKNOWN"

    # NOTE: You must configure credentials via environment variables or os.environ
    s3 = boto3.client("s3")
    file_name = f"bronze/sec_data_{ticker}_{cik}.json"

    s3.put_object(
        Bucket=bucket_name,
        Key=file_name,
        Body=json.dumps(data),
        ContentType="application/json",
    )
    return data.get("name", "Unknown Company")

def handler(event, context):
    user_email = os.environ.get("SEC_EMAIL", "anda@email.com")
    bucket_name = "config-elt-bucket"
    specific_cik = event.get("cik")

    try:
        if specific_cik:
            cik_padded = str(specific_cik).zfill(10)
            company_name = fetch_and_save_to_s3(cik_padded, user_email, bucket_name)
            return {"statusCode": 200, "body": json.dumps({"message": f"Sukses ingest {company_name}"})}
        else:
            ciks = get_all_nasdaq_ciks(user_email)
            if not ciks: raise Exception("Daftar CIK Nasdaq kosong.")

            target_ciks = ciks[:10]
            ingested = []
            for cik in target_ciks:
                name = fetch_and_save_to_s3(cik, user_email, bucket_name)
                ingested.append(name)
                time.sleep(0.15)

            return {"statusCode": 200, "body": json.dumps({"companies": ingested})}
    except Exception as e:
        print(f"Error: {str(e)}")
        return {"statusCode": 500, "body": json.dumps({"error": str(e)})}

In [ ]:
from google.colab import userdata
import os

try:
    # Load AWS credentials from Colab Secrets
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')

    # Optionally set region if needed (e.g., 'us-east-1')
    # os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

    event = {}
    context = None

    result = handler(event, context)
    print(result)
except userdata.SecretNotFoundError:
    print("Error: AWS secrets not found. Please add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY to the Secrets (key icon) sidebar.")

Error: AWS secrets not found. Please add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY to the Secrets (key icon) sidebar.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
import os

# Authenticate to Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the spreadsheet by path
try:
    path = "/content/drive/MyDrive/Colab Notebooks/RAKSA-eia-GH_accessKeys (1).gsheet"
    # Note: gspread usually needs the title or URL, but we can try opening the file directly or reading via pandas
    # Since it is a .gsheet, reading it as a dataframe is often easier via the title
    spreadsheet = gc.open("RAKSA-eia-GH_accessKeys (1)")
    sheet = spreadsheet.get_worksheet(0)
    df_keys = pd.DataFrame(sheet.get_all_records())

    # Extract keys (assuming standard column names 'Access key ID' and 'Secret access key')
    # Adjust column names if they are different in your sheet
    aws_access_key = df_keys.iloc[0]['Access key ID']
    aws_secret_key = df_keys.iloc[0]['Secret access key']

    os.environ['AWS_ACCESS_KEY_ID'] = str(aws_access_key)
    os.environ['AWS_SECRET_ACCESS_KEY'] = str(aws_secret_key)

    print("Credentials successfully loaded from Google Sheet.")

    # Run the handler
    event = {}
    context = None
    result = handler(event, context)
    print(result)

except Exception as e:
    print(f"Error loading credentials: {e}")
    print("Please ensure the column names in your Sheet match 'Access key ID' and 'Secret access key'.")

Credentials successfully loaded from Google Sheet.
{'statusCode': 200, 'body': '{"companies": ["NVIDIA CORP", "Alphabet Inc.", "Apple Inc.", "MICROSOFT CORP", "AMAZON COM INC", "Broadcom Inc.", "Tesla, Inc.", "Meta Platforms, Inc.", "MICRON TECHNOLOGY INC", "Walmart Inc."]}'}
